# 06 Trade / Skip Training

ИИ не выбирает направление заново. Direction берется из event detector, а модель решает: входить или пропустить.

In [1]:
from pathlib import Path
import random
import sys

import joblib
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src import config
from src.connector.data_fetcher import load_all_price_data
from src.features.event_detector import detect_events
from src.features.feature_pipeline import generate_features
from src.models.sequence_models import load_model_with_config
from src.models.trade_skip_training import (
    build_trade_skip_frame,
    generate_trade_skip_signal_history,
    train_trade_skip_model,
)
from src.strategy.backtest import build_trades, calculate_trade_metrics
from src.strategy.signal_generator import generate_rule_based_signal_history

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

pd.set_option("display.max_columns", None)

## Настройки

Главная идея: rule/event detector дает BUY/SELL, а AI фильтрует плохие входы.

In [2]:
RUN_TRAINING = True
MODEL_TYPES = ["gru", "lstm"]

SELECTION_METRIC = "balanced_accuracy"
EPOCHS = 40
Q_CANDLES = 2000
MIN_TRADES = 25

HORIZON = config.DEFAULT_HORIZON_CANDLES
TP_THRESHOLD = config.DEFAULT_TP_THRESHOLD
SL_THRESHOLD = config.DEFAULT_SL_THRESHOLD

THRESHOLD_GRID = np.round(np.arange(0.45, 0.76, 0.02), 2)
VALID_START = pd.Timestamp(config.TRAIN_END_DATE)
TEST_START = pd.Timestamp(config.VALID_END_DATE)

TRADE_SKIP_FEATURE_COLUMNS = [
    "event_cusum_direction",
    "near_support",
    "near_resistance",
    "breakout_up",
    "breakout_down",
    "strong_range",
    "strong_body",
    "range_ratio_20",
    "body_ratio_20",
    "dist_to_prev_sup",
    "dist_to_prev_res",
    "dist_to_prev_sup_96",
    "dist_to_prev_res_96",
    "rsi_14_norm",
    "volatility_20",
    "volatility_96",
    "time_sin",
    "time_cos",
    "dow_sin",
    "dow_cos",
]

PARAMS = {
    "gru": {
        "learning_rate": 0.0015,
        "hidden_size": 64,
        "dropout": 0.25,
        "num_layers": 2,
        "batch_size": 128,
    },
    "lstm": {
        "learning_rate": 0.0015,
        "hidden_size": 64,
        "dropout": 0.30,
        "num_layers": 2,
        "batch_size": 128,
    },
}

In [3]:
# Загружаем данные и считаем базовые события.
price_df, loaded_files = load_all_price_data(PROJECT_ROOT / "data")
prepared_df = detect_events(generate_features(price_df))
trade_skip_df = build_trade_skip_frame(
    price_df,
    horizon=HORIZON,
    tp_threshold=TP_THRESHOLD,
    sl_threshold=SL_THRESHOLD,
    feature_columns=TRADE_SKIP_FEATURE_COLUMNS,
)

print(f"CSV файлов: {len(loaded_files)}")
print(f"Свечей: {len(price_df):,}")
print(f"Events: {int(prepared_df['event'].sum()):,}")
print("Trade success balance:")
display(trade_skip_df.loc[trade_skip_df['trade_success'].notna(), 'trade_success'].value_counts(normalize=True).rename('share'))

CSV файлов: 45
Свечей: 277,105
Events: 21,684
Trade success balance:


trade_success
0.0    0.534219
1.0    0.465781
Name: share, dtype: float64

## Обучение

Модель учится отвечать: даст ли сделка по event direction положительный результат по текущим TP/SL/HORIZON.

In [4]:
def trade_skip_paths(model_type: str):
    model_path = config.MODELS_DIR / f"trade_skip_event_{model_type}_best.pth"
    scaler_path = config.MODELS_DIR / f"trade_skip_event_{model_type}_scaler.pkl"
    config_path = config.MODELS_DIR / f"trade_skip_event_{model_type}_config.pkl"
    return model_path, scaler_path, config_path


def train_one(model_type: str):
    params = PARAMS[model_type]
    model_path, scaler_path, config_path = trade_skip_paths(model_type)
    result = train_trade_skip_model(
        price_df=price_df,
        model_type=model_type,
        horizon=HORIZON,
        tp_threshold=TP_THRESHOLD,
        sl_threshold=SL_THRESHOLD,
        epochs=EPOCHS,
        selection_metric=SELECTION_METRIC,
        model_path=model_path,
        scaler_path=scaler_path,
        feature_columns=TRADE_SKIP_FEATURE_COLUMNS,
        **params,
    )
    joblib.dump(
        {
            "task": "trade_skip",
            "model_type": model_type,
            "input_size": len(TRADE_SKIP_FEATURE_COLUMNS),
            "hidden_size": params["hidden_size"],
            "dropout": params["dropout"],
            "num_layers": params["num_layers"],
            "selection_metric": SELECTION_METRIC,
            "horizon": HORIZON,
            "tp_threshold": TP_THRESHOLD,
            "sl_threshold": SL_THRESHOLD,
            "feature_columns": TRADE_SKIP_FEATURE_COLUMNS,
        },
        config_path,
    )
    return {
        "model": model_type,
        "best_valid_score": result["best_valid_score"],
        "test_accuracy": result["test_metrics"]["accuracy"],
        "test_balanced_accuracy": result["test_metrics"]["balanced_accuracy"],
        "test_f1": result["test_metrics"]["f1"],
        "test_precision": result["test_metrics"]["precision"],
        "test_recall": result["test_metrics"]["recall"],
        "train_samples": result["train_samples"],
        "valid_samples": result["valid_samples"],
        "test_samples": result["test_samples"],
        "model_path": str(model_path),
        "config_path": str(config_path),
    }

In [5]:
if RUN_TRAINING:
    train_rows = [train_one(model_type) for model_type in MODEL_TYPES]
    display(pd.DataFrame(train_rows))
else:
    print("Обучение пропущено, используем сохраненные trade_skip модели.")

,model,best_valid_score,test_accuracy,test_balanced_accuracy,test_f1,test_precision,test_recall,train_samples,valid_samples,test_samples,model_path,config_path
0,gru,0.512834,0.504412,0.514693,0.598268,0.489526,0.769119,11517,5856,4306,/workspace/data/models/trade_skip_event_gru_be...,/workspace/data/models/trade_skip_event_gru_co...
1,lstm,0.517035,0.505341,0.516545,0.606285,0.490431,0.793804,11517,5856,4306,/workspace/data/models/trade_skip_event_lstm_b...,/workspace/data/models/trade_skip_event_lstm_c...


## Validation threshold

Threshold подбираем на validation, test не трогаем до финальной проверки.

In [6]:
def localize_like_index(ts: pd.Timestamp, index: pd.Index) -> pd.Timestamp:
    if getattr(index, "tz", None) is not None and ts.tzinfo is None:
        return ts.tz_localize(index.tz)
    return ts


VALID_START_TS = localize_like_index(VALID_START, prepared_df.index)
TEST_START_TS = localize_like_index(TEST_START, prepared_df.index)


def select_period(signals: pd.DataFrame, start: pd.Timestamp, end: pd.Timestamp | None = None, q_candles: int | None = None) -> pd.DataFrame:
    result = signals[signals["time"].ge(start)].copy()
    if end is not None:
        result = result[result["time"].lt(end)].copy()
    if q_candles is not None:
        result = result.tail(q_candles).copy()
    return result


def summarize_strategy(name: str, signals: pd.DataFrame, threshold: float) -> dict:
    if "probability_trade" in signals.columns:
        filtered = signals.copy()
        can_trade = filtered["event"].eq(1) & filtered["probability_trade"].ge(threshold)
        filtered["decision"] = "NO TRADE"
        filtered.loc[can_trade & filtered["event_cusum_direction"].eq(1), "decision"] = "BUY"
        filtered.loc[can_trade & filtered["event_cusum_direction"].eq(-1), "decision"] = "SELL"
    else:
        filtered = signals.copy()

    trades = build_trades(filtered, prepared_df, horizon=HORIZON, tp_threshold=TP_THRESHOLD, sl_threshold=SL_THRESHOLD)
    metrics = calculate_trade_metrics(trades)
    return {
        "strategy": name,
        "threshold": threshold,
        "trades": metrics["Trades"],
        "total_return": metrics["Total Return"],
        "winrate": metrics["Win Rate"],
        "profit_factor": metrics["Profit Factor"],
        "max_drawdown": metrics["Max Drawdown"],
        "avg_trade": metrics["Average Trade"],
    }


def selection_score(row: pd.Series) -> float:
    score = row["total_return"]
    if row["trades"] < MIN_TRADES:
        score -= 10.0
    if row["profit_factor"] < 1.0:
        score -= 1.0
    return score

In [7]:
# Генерируем сырые probability_trade для всего периода.
raw_by_model = {}
for model_type in MODEL_TYPES:
    model_path, scaler_path, config_path = trade_skip_paths(model_type)
    model_config = joblib.load(config_path)
    model = load_model_with_config(model_path, config_path)
    scaler = joblib.load(scaler_path)
    raw = generate_trade_skip_signal_history(
        price_df,
        model=model,
        scaler=scaler,
        feature_columns=model_config["feature_columns"],
        threshold=0.0,
        max_rows=len(prepared_df),
    )
    raw_by_model[model_type] = {
        "valid": select_period(raw, VALID_START_TS, TEST_START_TS),
        "test": select_period(raw, TEST_START_TS, None, Q_CANDLES),
    }
    print(model_type, "valid", len(raw_by_model[model_type]["valid"]), "test", len(raw_by_model[model_type]["test"]))

gru valid 73880 test 2000
lstm valid 73880 test 2000


In [8]:
valid_rows = []
for model_type, periods in raw_by_model.items():
    for threshold in THRESHOLD_GRID:
        valid_rows.append(summarize_strategy(f"TradeSkip + {model_type.upper()}", periods["valid"], float(threshold)))

valid_df = pd.DataFrame(valid_rows)
valid_df["selection_score"] = valid_df.apply(selection_score, axis=1)
valid_top = valid_df.sort_values(["selection_score", "profit_factor", "trades"], ascending=[False, False, False]).head(20)
display(valid_top)

,strategy,threshold,trades,total_return,winrate,profit_factor,max_drawdown,avg_trade,selection_score
21,TradeSkip + LSTM,0.55,257,-0.01627,0.459144,0.857431,-0.027230,-0.000063,-1.01627
4,TradeSkip + GRU,0.53,674,-0.04981,0.448071,0.835838,-0.057008,-0.000074,-1.04981
20,TradeSkip + LSTM,0.53,1783,-0.12185,0.455973,0.850866,-0.125299,-0.000068,-1.12185
19,TradeSkip + LSTM,0.51,3632,-0.16723,0.468888,0.894811,-0.179986,-0.000046,-1.16723
3,TradeSkip + GRU,0.51,3726,-0.16858,0.470209,0.894874,-0.177580,-0.000045,-1.16858
2,TradeSkip + GRU,0.49,4689,-0.24696,0.463638,0.877437,-0.256265,-0.000053,-1.24696
1,TradeSkip + GRU,0.47,5157,-0.27901,0.463060,0.873601,-0.288049,-0.000054,-1.27901
18,TradeSkip + LSTM,0.49,5037,-0.30801,0.459003,0.859296,-0.320666,-0.000061,-1.30801
0,TradeSkip + GRU,0.45,5447,-0.31162,0.460437,0.866575,-0.321438,-0.000057,-1.31162
17,TradeSkip + LSTM,0.47,5552,-0.32897,0.459834,0.862684,-0.338624,-0.000059,-1.32897


## Финальный test

Берем лучший threshold с validation и сравниваем с rule baseline на последних Q_CANDLES свечах test.

In [9]:
best_by_strategy = (
    valid_df.sort_values(["selection_score", "profit_factor"], ascending=[False, False])
    .groupby("strategy", as_index=False)
    .head(1)
)

test_rows = []
for _, row in best_by_strategy.iterrows():
    model_type = row["strategy"].split("+")[-1].strip().lower()
    test_rows.append(summarize_strategy(row["strategy"], raw_by_model[model_type]["test"], float(row["threshold"])))

rule_signals = generate_rule_based_signal_history(price_df, max_rows=Q_CANDLES)
test_rows.append(summarize_strategy("Rule baseline", rule_signals, 0.0))

test_df = pd.DataFrame(test_rows).sort_values(["total_return", "profit_factor"], ascending=[False, False])
display(test_df)

,strategy,threshold,trades,total_return,winrate,profit_factor,max_drawdown,avg_trade
2,Rule baseline,0.00,163,0.01640,0.558282,1.259864,-0.009771,0.000101
1,TradeSkip + GRU,0.53,18,0.00379,0.611111,1.624382,-0.001841,0.000211
0,TradeSkip + LSTM,0.55,4,-0.00146,0.250000,0.391667,-0.002398,-0.000365


## Следующие варианты

Если TradeSkip не обгонит rule baseline, дальше стоит подбирать TP/SL/HORIZON и добавить event types: sweep, displacement, breakout-retake.